In [1]:
import chromadb

In [2]:
# Persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [ ]:
chroma_client.list_collections()
#chroma_client.delete_collection(name = "origin")

In [28]:
collection = chroma_client.create_collection(name = "origin")

In [9]:
from pypdf import PdfReader

reader = PdfReader("gutenberg_origin1228_pg1228.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()


In [10]:
def split_text(text, chunk_size=500):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

chunks = split_text(text)

In [17]:
chunks[0:100]

['The Project Gutenberg eBook of On the Origin of Species By Means of Natural Selection\n    \nThis eBook is for the use of anyone anywhere in the United States and\nmost other parts of the world at no cost and with almost no restrictions\nwhatsoever. You may copy it, give it away or re-use it under the terms\nof the Project Gutenberg License included with this eBook or online\nat www.gutenberg.org. If you are not located in the United States,\nyou will have to check the laws of the country where you are',
 ' located\nbefore using this eBook.\nTitle: On the Origin of Species By Means of Natural Selection\nAuthor: Charles Darwin\n        \nRelease date: March 1, 1998 [eBook #1228]\n                Most recently updated: October 29, 2024\nLanguage: English\nOther information and formats: www.gutenberg.org/ebooks/1228\nCredits: Sue Asscher and David Widger\n*** START OF THE PROJECT GUTENBERG EBOOK ON THE ORIGIN OF SPECIES BY MEANS OF NATURAL SELECTION ***\nThere are several editions of th

In [12]:
import os
from openai import OpenAI

In [19]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

In [ ]:
# client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
#                 api_key='any value',
#                 default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# response = client.embeddings.create(
#     input = chunks[0:10], 
#     model = "text-embedding-3-small"
# )
# response.data


[Embedding(embedding=[0.08551025390625, 0.0430908203125, 0.057830810546875, 0.0467529296875, -0.011932373046875, -0.0162506103515625, -0.0469970703125, -0.0128021240234375, -0.047454833984375, -0.03228759765625, 0.049163818359375, -0.032928466796875, -0.02691650390625, 0.00873565673828125, 0.024444580078125, 0.0024280548095703125, 0.053436279296875, -0.0300445556640625, -0.00983428955078125, 0.043975830078125, 0.019561767578125, -0.010955810546875, -0.043670654296875, 0.0087738037109375, -0.050628662109375, -0.050811767578125, -0.0237274169921875, 0.0240936279296875, 0.027740478515625, 0.010772705078125, -0.05047607421875, -0.034027099609375, -0.0193328857421875, -0.01123046875, -0.00799560546875, 0.0122528076171875, -0.004032135009765625, -0.03887939453125, 0.005893707275390625, -0.01824951171875, -0.0261383056640625, -0.13818359375, 0.031036376953125, 0.056060791015625, -0.033782958984375, 0.01306915283203125, 0.02935791015625, -0.05096435546875, 0.00189971923828125, 0.00208854675292

In [21]:
embeddings = [item.embedding for item in response.data]

In [23]:
for i, chunk in enumerate(chunks[0:10]):
    collection.add(
        documents=[chunk],
        embeddings=[embeddings[i]],
        ids=[str(i)]
    )

In [29]:
def batchify(data, batch_size=50):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

In [30]:
for i, batch in enumerate(batchify(chunks, 50)):
    print(f"Processing batch {i+1}")

Processing batch 1
Processing batch 2
Processing batch 3
Processing batch 4
Processing batch 5
Processing batch 6
Processing batch 7
Processing batch 8
Processing batch 9
Processing batch 10
Processing batch 11
Processing batch 12
Processing batch 13
Processing batch 14
Processing batch 15
Processing batch 16
Processing batch 17
Processing batch 18
Processing batch 19
Processing batch 20
Processing batch 21
Processing batch 22
Processing batch 23
Processing batch 24
Processing batch 25
Processing batch 26
Processing batch 27
Processing batch 28
Processing batch 29
Processing batch 30
Processing batch 31
Processing batch 32
Processing batch 33
Processing batch 34
Processing batch 35
Processing batch 36
Processing batch 37
Processing batch 38
Processing batch 39


In [31]:
len(chunks)

1902

In [ ]:
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                 api_key='any value',
                 default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [32]:
for i, batch in enumerate(batchify(chunks, 50)):
    print(f"Processing batch {i+1}")
    response = client.embeddings.create(
        input=batch,
        model="text-embedding-3-small"
    )
    
    embeddings = [d.embedding for d in response.data]

    collection.add(
        documents=batch,
        embeddings=embeddings,
        ids=[f"{i}_{j}" for j in range(len(batch))]
    )

Processing batch 1
Processing batch 2
Processing batch 3
Processing batch 4
Processing batch 5
Processing batch 6
Processing batch 7
Processing batch 8
Processing batch 9
Processing batch 10
Processing batch 11
Processing batch 12
Processing batch 13
Processing batch 14
Processing batch 15
Processing batch 16
Processing batch 17
Processing batch 18
Processing batch 19
Processing batch 20
Processing batch 21
Processing batch 22
Processing batch 23
Processing batch 24
Processing batch 25
Processing batch 26
Processing batch 27
Processing batch 28
Processing batch 29
Processing batch 30
Processing batch 31
Processing batch 32
Processing batch 33
Processing batch 34
Processing batch 35
Processing batch 36
Processing batch 37
Processing batch 38
Processing batch 39


In [34]:
def search_pdf(query: str):
    query_embedding = client.embeddings.create(
        input=query,
        model="text-embedding-3-small"
    ).data[0].embedding

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    return results["documents"][0]

In [35]:
query = "What is natural selection?"

In [36]:
def generate_response(query:str):
    context = search_pdf(query)
    print("Generated context:\n", context)
    prompt = f"""
    Answer the question using ONLY the context below.
    If the answer is not in the context, say "I don't know".
    Context:
    {context}

    Question:
    {query}
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that provides information based on book context."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        temperature=0.7
    )
    return response.choices[0].message.content

In [ ]:
answer = generate_response(
    "What is natural selection?")


Generated context:
 ['nd to external nature, will tend to the preservation of that\nindividual, and will generally be inherited by its offspring. The\noffspring, also, will thus have a better chance of surviving, for, of\nthe many individuals of any species which are periodically born, but a\nsmall number can survive. I have called this principle, by which each\nslight variation, if useful, is preserved, by the term of Natural\nSelection, in order to mark its relation to man’s power of selection.\nWe have seen that man by s', 'ariations and the rejection of injurious variations, I call Natural\nSelection. Variations neither useful nor injurious would not be\naffected by natural selection, and would be left a fluctuating element,\nas perhaps we see in the species called polymorphic.\nWe shall best understand the probable course of natural selection by\ntaking the case of a country undergoing some physical change, for\ninstance, of climate. The proportional numbers of its inhabitants wou

In [ ]:

print("Answer from LLM:",answer)